## load raw data from azure datalake via sparkR

In [0]:
%r

## SPARKR TO ACCESS DATALAKE AND READ PARQUET FILES
library(dplyr)
library(lubridate)
library(ggplot2)
library(SparkR)
sparkR.session()

## READ IN PARQUET FILES TO SPARK DATAFRAMES
df <- read.parquet( "vascular/CONDITION_OCCURRENCE.parquet") 
df2 <- read.parquet( "vascular/PERSON.parquet") 
df3 <- read.parquet( "vascular/CONCEPT.parquet") 
df4 <- read.parquet( "vascular/DRUG_EXPOSURE.parquet") 
df5 <- read.parquet( "vascular/MEASUREMENT.parquet") 
df6 <- read.parquet( "vascular/OBSERVATION.parquet") 
df7 <- read.parquet( "vascular/PROCEDURE_OCCURRENCE.parquet") 
df8 <- read.parquet( "vascular/VISIT_DETAIL.parquet") 
df9 <- read.parquet( "vascular/VISIT_OCCURRENCE.parquet") 
df10 <- read.parquet( "vascular/STATE_DEATH_DATA.parquet") 
df11 <- read.parquet( "vascular/PVL_MEASUREMENTS.parquet")


##CREATE DELTA TABLE TEMP VIEWS FOR SQL

#createOrReplaceTempView(df, "condition")
#createOrReplaceTempView(df2, "person")
#createOrReplaceTempView(df3, "concept")
#createOrReplaceTempView(df4, "drug_exposure")
#createOrReplaceTempView(df5, "measurement")
#createOrReplaceTempView(df6, "observation")
#createOrReplaceTempView(df7, "procedure_occurrence")
#createOrReplaceTempView(df8, "visit_detail")
#createOrReplaceTempView(df9, "visit_occurrence")
#createOrReplaceTempView(df10, "death")


In [0]:
%r
##WRITE DATAFRAMES TO DELTA TABLES

write.df(df, mode = 'overwrite', path = 'hive.db/vascular_condition', source = 'delta')
write.df(df2, mode = 'overwrite', path = 'hive.db/vascular_person', source = 'delta')
write.df(df3, mode = 'overwrite', path = 'hive.db/vascular_concept', source = 'delta')
write.df(df4, mode = 'overwrite', path = 'hive.db/vascular_drug_exposure', source = 'delta')
write.df(df5, mode = 'overwrite', path = 'hive.db/vascular_measurement', source = 'delta')
write.df(df6, mode = 'overwrite', path = 'hive.db/vascular_observation', source = 'delta')
write.df(df7, mode = 'overwrite', path = 'hive.db/vascular_procedure_occurrence', source = 'delta')
write.df(df8, mode = 'overwrite', path = 'hive.db/vascular_visit_detail', source = 'delta')
write.df(df9, mode = 'overwrite', path = 'hive.db/vascular_visit_occurrence', source = 'delta')
write.df(df10, mode = 'overwrite', path = 'hive.db/vascular_state_death_data', source = 'delta')
write.df(df11, mode = 'overwrite', path = 'hive.db/vascular_pvl_measurements', source = 'delta')


In [0]:
--CREATE TABLES IN METASTORE SO THEY'RE AVAILABLE IN THE UNITY CATALOG

CREATE TABLE IF NOT EXISTS nctracs.vascular_condition USING DELTA LOCATION 'hive.db/vascular_condition';
CREATE TABLE IF NOT EXISTS nctracs.vascular_person USING DELTA LOCATION 'hive.db/vascular_person';
CREATE TABLE IF NOT EXISTS nctracs.vascular_concept USING DELTA LOCATION 'hive.db/vascular_concept';
CREATE TABLE IF NOT EXISTS nctracs.vascular_drug_exposure USING DELTA LOCATION 'hive.db/vascular_drug_exposure';
CREATE TABLE IF NOT EXISTS nctracs.vascular_measurement USING DELTA LOCATION 'hive.db/vascular_measurement';
CREATE TABLE IF NOT EXISTS nctracs.vascular_observation USING DELTA LOCATION 'hive.db/vascular_observation';
CREATE TABLE IF NOT EXISTS nctracs.vascular_procedure_occurrence USING DELTA LOCATION 'hive.db/vascular_procedure_occurrence';
CREATE TABLE IF NOT EXISTS nctracs.vascular_visit_detail USING DELTA LOCATION 'hive.db/vascular_visit_detail';
CREATE TABLE IF NOT EXISTS nctracs.vascular_visit_occurrence USING DELTA LOCATION 'hive.db/vascular_visit_occurrence';
CREATE TABLE IF NOT EXISTS nctracs.vascular_state_death_data USING DELTA LOCATION 'hive.db/vascular_state_death_data';
CREATE TABLE IF NOT EXISTS nctracs.vascular_pvl_measurements USING DELTA LOCATION 'hive.db/vascular_pvl_measurements';

In [0]:
--check how many patients are in the vascular dataset
select count(*) from nctracs.vascular_person

In [0]:
--CREATE CONCEPT_FREQS TABLE BY UNION

drop table if exists nctracs.vascular_concept_freqs;

create table nctracs.vascular_concept_freqs as (
select 'condition' as domain, count(CONDITION_CONCEPT_ID) as freq, concept_id, concept_name, CONDITION_SOURCE_VALUE 
from nctracs.vascular_condition a LEFT JOIN nctracs.vascular_concept b on a.CONDITION_CONCEPT_ID = b.CONCEPT_ID
group by concept_id, concept_name, condition_source_value

UNION ALL

select 'procedure' as domain, count(procedure_concept_id) as freq, concept_id, concept_name, procedure_source_value
from nctracs.vascular_procedure_occurrence a LEFT JOIN nctracs.vascular_concept b on a.procedure_concept_id = b.CONCEPT_ID
group by concept_id, concept_name, procedure_source_value

UNION ALL

select 'observation' as domain, count(observation_concept_id) as freq, concept_id, concept_name, observation_source_value
from nctracs.vascular_observation a LEFT JOIN nctracs.vascular_concept b on a.observation_concept_id = b.CONCEPT_ID
group by concept_id, concept_name, observation_source_value

UNION ALL

select 'drug' as domain, count(drug_concept_id) as freq, concept_id, concept_name, drug_source_value
from nctracs.vascular_drug_exposure a LEFT JOIN nctracs.vascular_concept b on a.drug_concept_id = b.CONCEPT_ID
group by concept_id, concept_name, drug_source_value

UNION ALL

select 'measurement' as domain, count(measurement_concept_id) as freq, concept_id, concept_name, measurement_source_value
from nctracs.vascular_measurement a LEFT JOIN nctracs.vascular_concept b on a.measurement_concept_id = b.CONCEPT_ID
group by concept_id, concept_name, measurement_source_value

UNION ALL

select 'visit_detail' as domain, count(visit_detail_concept_id) as freq, concept_id, concept_name, visit_detail_source_value
from nctracs.vascular_visit_detail a LEFT JOIN nctracs.vascular_concept b on a.visit_detail_concept_id = b.CONCEPT_ID
group by concept_id, concept_name, visit_detail_source_value
)

In [0]:
CREATE TABLE IF NOT EXISTS nctracs.vascular_concept_ancestors AS
SELECT DISTINCT
  c1.domain_id AS ancestor_domain,
  ca.ancestor_concept_id,
  c1.concept_name AS ancestor,
  cf1.freq AS ancestor_freq,
  '---------------' AS buffer,
  c2.domain_id AS descendant_domain,
  ca.descendant_concept_id,
  c2.concept_name AS descendant,
  cf2.freq AS descendant_freq,
  ca.min_levels_of_separation,
  ca.max_levels_of_separation
FROM nctracs_environmental.concept_ancestor ca
LEFT JOIN nctracs.vascular_concept c1 ON ca.ancestor_concept_id = c1.concept_id
LEFT JOIN nctracs.vascular_concept c2 ON ca.descendant_concept_id = c2.concept_id
LEFT JOIN nctracs.vascular_concept_freqs cf1 ON ca.ancestor_concept_id = cf1.concept_id
LEFT JOIN nctracs.vascular_concept_freqs cf2 ON ca.descendant_concept_id = cf2.concept_id
WHERE ca.ancestor_concept_id != ca.descendant_concept_id
AND NOT (cf1.freq IS NULL AND cf2.freq IS NULL) 
